In [0]:
from pyspark.sql.functions import col, lower, upper, trim, when, current_timestamp, coalesce, lit

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

print("Silver schema ready")

In [0]:
regions = (
    spark.table("azure_blob_storage.src_regions")
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("region_name", coalesce(lower(trim(col("region_name"))), lit("none")))
    .withColumn("country", 
        when(upper(trim(col("country"))).isin(["USA", "U.S.A", "U.S.A.", "UNITED STATES"]), "USA")
        .when(upper(trim(col("country"))) == "INDIA", "INDIA")
        .otherwise(coalesce(upper(trim(col("country"))), lit("NONE")))
    )
    .withColumn("ingestion_ts", current_timestamp())
    .dropDuplicates(["region_code"])
)

regions.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.regions")
print(f"regions: {regions.count()} records")

In [0]:
print("\nAll regions:")
display(spark.table("silver.regions"))